# Exercício 12 — Classificação

Este exercício repete o pipeline da aula na **sua coleta** (a mesma das Aulas 5 a 7 e do Exercício 11). Você vai transformar `plays` num rótulo de "viralizou", treinar um classificador e avaliá-lo com matriz de confusão, não só com acurácia.

Copie a sua `exportacao.csv` para `dados/exportacao.csv` nesta pasta. Antes de tudo, copie a pasta `exercicios/` para dentro da sua pasta de entregas (`extracao-dados-trabalhos-seunome`), numa pasta `12-classificacao` dentro de `projetos/`.

## Preparação do ambiente

Dentro da pasta, no Windows (Prompt de Comando ou Terminal integrado do VS Code):

```cmd
uv venv .venv
uv pip install -r requirements.txt
```

No Mac (Terminal), os mesmos comandos. Se o `uv` não funcionar, `pip install -r requirements.txt` com o ambiente ativado.

## Parte 0 — Dados e a decisão do rótulo

**Fonte dos dados:** copie a sua coleta para `dados/exportacao.csv`.

**Como você vai definir "viralizou" (preencha antes de codar):**

> Defini 'viralizou' com base em plays e percentil 90 como corte, porque plays é a métrica com dados mais completos e estáveis da coleta (ao contrário de curtidas, comentários e compartilhamentos, que são muito mais raros e deixariam a classe viral pequena demais para o modelo aprender), e o percentil 90 separa um grupo de 'viral' pequeno o suficiente para ser um evento raro de verdade, mas com posts positivos suficientes (140) para o modelo ter exemplos para aprender."

## Parte 1 — Carregar, montar features e o rótulo

Ajuste os nomes de coluna se a sua plataforma for diferente. **Nada de vazamento:** `likes`/`comments`/`shares`/`plays` não entram no `X`.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

df = pd.read_csv("/Users/leonardorosa/estudo/extracao-dados-trabalhos-juliacereja/projetos/12-classificacao/dados/exportacao.csv", sep=";")
df = df.drop_duplicates()
df = df[df["plays"] > 0].copy()

# features (reaproveite/adapte as da aula)
X = pd.DataFrame(index=df.index)
# complete aqui: crie pelo menos TRÊS features (pode copiar do Exercício 11)
X["seguidores_autor"] = df["author_followers"]
X["tam_legenda"] = df["body"].fillna("").str.len()
X["n_hashtags"] = df["hashtags"].fillna("").apply(lambda s: 0 if s == "" else len(s.split(",")))
momento = pd.to_datetime(df["timestamp"])
X["hora"] = momento.dt.hour


print("Features:", list(X.columns))

Features: ['seguidores_autor', 'tam_legenda', 'n_hashtags', 'hora']


In [4]:
# complete aqui: crie o rótulo y (0/1) a partir do corte que você definiu na Parte 0
# exemplo com percentil 90:

corte = df["plays"].quantile(0.90)
y = (df["plays"] > corte).astype(int)

print(f"Marcados como 'viralizou': {y.sum()} de {len(y)}  ({y.mean():.0%})")

Marcados como 'viralizou': 140 de 1416  (10%)


## Parte 2 — Treino, teste e regressão logística

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_treino, y_treino)
decisao = modelo.predict(X_teste)
probabilidade = modelo.predict_proba(X_teste)[:, 1]

cm = confusion_matrix(y_teste, decisao)
print("Matriz de confusão [ [VN, FP], [FN, VP] ]:")
print(cm)
print(f"Acurácia {accuracy_score(y_teste, decisao):.2f} | "
      f"Precisão {precision_score(y_teste, decisao, zero_division=0):.2f} | "
      f"Recall {recall_score(y_teste, decisao, zero_division=0):.2f} | "
      f"F1 {f1_score(y_teste, decisao, zero_division=0):.2f}")

Matriz de confusão [ [VN, FP], [FN, VP] ]:
[[317   2]
 [ 33   2]]
Acurácia 0.90 | Precisão 0.50 | Recall 0.06 | F1 0.10


## Parte 3 — Testar dois thresholds

Complete: para pelo menos **dois** cortes diferentes de 0,5 (ex.: 0,30 e 0,20), recalcule precisão, recall e F1 usando `probabilidade`. Imprima os resultados lado a lado com o corte 0,50.

In [6]:
# complete aqui: um laço sobre alguns valores de threshold, aplicando (probabilidade >= t)
for t in [0.50, 0.30, 0.20, 0.15]:
    decisao_t = (probabilidade >= t).astype(int)
    p = precision_score(y_teste, decisao_t, zero_division=0)
    r = recall_score(y_teste, decisao_t, zero_division=0)
    f = f1_score(y_teste, decisao_t, zero_division=0)
    vp = int(((decisao_t == 1) & (y_teste.values == 1)).sum())
    fp = int(((decisao_t == 1) & (y_teste.values == 0)).sum())
    print(f"corte {t:.2f}:  precisão {p:.2f}  recall {r:.2f}  F1 {f:.2f}   (pegou {vp} virais, {fp} alarmes falsos)")

corte 0.50:  precisão 0.50  recall 0.06  F1 0.10   (pegou 2 virais, 2 alarmes falsos)
corte 0.30:  precisão 0.50  recall 0.09  F1 0.15   (pegou 3 virais, 3 alarmes falsos)
corte 0.20:  precisão 0.18  recall 0.09  F1 0.12   (pegou 3 virais, 14 alarmes falsos)
corte 0.15:  precisão 0.27  recall 0.23  F1 0.25   (pegou 8 virais, 22 alarmes falsos)


## Parte 4 — Uma árvore, para comparar

Complete: treine uma `DecisionTreeClassifier(max_depth=4, random_state=42, class_weight="balanced")`, preveja em `X_teste`, calcule precisão/recall/F1 e imprima as `feature_importances_` ordenadas.

In [7]:
from sklearn.tree import DecisionTreeClassifier

# complete aqui
arvore = DecisionTreeClassifier(max_depth=4, random_state=42, class_weight="balanced")
arvore.fit(X_treino, y_treino)
decisao_arvore = arvore.predict(X_teste)

print("Árvore (profundidade 4, class_weight balanced):")
print(f"  precisão {precision_score(y_teste, decisao_arvore, zero_division=0):.2f}"
      f"  recall {recall_score(y_teste, decisao_arvore, zero_division=0):.2f}"
      f"  F1 {f1_score(y_teste, decisao_arvore, zero_division=0):.2f}")
print()

importancias = pd.DataFrame({
    "feature": X.columns,
    "importancia": arvore.feature_importances_,
}).sort_values("importancia", ascending=False)
importancias

Árvore (profundidade 4, class_weight balanced):
  precisão 0.18  recall 0.74  F1 0.29



,feature,importancia
0,seguidores_autor,0.627564
1,tam_legenda,0.166185
2,n_hashtags,0.138709
3,hora,0.067542


## Parte 5 — README de reprodução

Crie `README.md` dentro de `projetos/12-classificacao/` na sua pasta de entregas:

**Fonte e período dos dados; quantos posts entraram:**

> Escreva aqui.

**Como você definiu "viralizou", e por quê:**

> Escreva aqui (o corte e a justificativa).

**Quantos posts ficaram como "viral" (a classe é rara?):**

> Escreva aqui.

**Features usadas; colunas descartadas por vazamento:**

> Escreva aqui.

**Matriz de confusão do seu melhor modelo, e uma leitura: a favor de quem ele erra?** (deixa passar muitos virais = recall baixo; ou dá muito alarme falso = precisão baixa)

> Escreva aqui.

**O que mudou quando você baixou o threshold:**

> Escreva aqui.

**Declaração de uso de IA:** ferramenta usada, em que trecho ou decisão, e o que você conferiu ou alterou depois (mesmo que seja "não usei IA nesta entrega").

> Escreva aqui.

## Parte 6 — Conferência final

- [ ] `dados/exportacao.csv` é a sua coleta, e `dados/` está no `.gitignore`.
- [ ] O rótulo "viralizou" foi definido por um corte **justificado** no README.
- [ ] O `X` tem pelo menos três features sem vazamento.
- [ ] A matriz de confusão e precisão/recall/F1 estão calculadas, não só a acurácia.
- [ ] Pelo menos dois thresholds diferentes de 0,5 foram testados.
- [ ] Uma árvore foi treinada e comparada.
- [ ] O README responde todas as perguntas da Parte 5, incluindo a leitura "a favor de quem o modelo erra".
- [ ] O notebook roda do início ao fim com Kernel → Restart e Run All.
- [ ] O README registra o uso de IA (ou informa que não houve).
- [ ] Notebook e README copiados em `projetos/12-classificacao/` na pasta de entregas.
- [ ] Você já fez `git add`, `git commit` e `git push` dessa entrega.